In [1]:
# pip install torch-geometric -q
# pip install scikit-learn scipy numpy pandas lightgbm -q

In [1]:
import subprocess
import sys
import torch
import numpy as np
import random
from torch_geometric.nn import GCNConv

# # Install required packages
# subprocess.check_call([sys.executable, "-m", "pip", "install",
#                        "torch-geometric", "torch-scatter", "torch-sparse"])

torch.cuda.is_available() 
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0))
mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"Total memory: {mem:.1f} GB")


# Set seeds
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

GPU: NVIDIA L4
Total memory: 22.1 GB


In [2]:
import pandas as pd

# Load preprocessed features 590540 rows × 633 columns = trainID + 631 + isFraud 
train_features = pd.read_csv(
    'ori_feature/train_features_v631.csv'
)

#632 columns = trainID + 631 
test_features = pd.read_csv(
    'ori_feature/test_features_v631.csv'
)

# # Load original train for labels and TransactionDT
# train_original = pd.read_csv('ieee-fraud-detection/train_transaction.csv')
# # load feature list
# test_original = pd.read_csv('ieee-fraud-detection/test_transaction.csv')

import json
with open('feature_cols.json', 'r') as f:
    feature_cols = json.load(f)

# Define edge columns (only keep those present in train_features)
edge_cols = [
    'card1', 'card2', 'card3', 'card5',
    'addr1', 'addr2','user_id1', 'user_id3']

# # Merge labels into train_features 590540 rows × 636 columns
# train_features = train_features.merge(
#     train_original[['TransactionID','card1', 'card2', 'addr1']],
#     on='TransactionID',
#     how='left'
# )
# #506691 rows × 635 columns
# test_features = test_features.merge(
#     test_original[['TransactionID','card1', 'card2', 'addr1']],
#     on='TransactionID',
#     how='left'
# )
# Verify
print(train_features.shape)
print(test_features.shape)
print("Fraud rate:", train_features['isFraud'].mean())
print("Columns:", train_features.columns.tolist())



edge_cols = [c for c in edge_cols if c in train_features.columns]

print(f"Feature dimensions: {len(feature_cols)}")
print(f"Edge columns available: {edge_cols}")

/tmp/ipykernel_979/225680317.py:4: DtypeWarning: Columns (0: id_23) have mixed types. Specify dtype option on import or set low_memory=False.
  train_features = pd.read_csv(
/tmp/ipykernel_979/225680317.py:9: DtypeWarning: Columns (0: id_23) have mixed types. Specify dtype option on import or set low_memory=False.
  test_features = pd.read_csv(


(590540, 638)
(506691, 637)
Fraud rate: 0.03499000914417313
Columns: ['TransactionID', 'TransactionDT', 'TransactionAmt', 'dist1', 'dist2', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V66', 'V67', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74', 'V75', 'V76', 'V77', 'V78', 'V79', 'V80', 'V81', 'V82', 'V83', 'V84', 'V85', 'V86', 'V87', 'V90', 'V91', 'V92', 'V93', 'V94', 'V95', 'V96', 'V97', 'V98', 'V99', 'V100', 'V101', 'V102', 'V103', 'V104', 

In [3]:
# Block 3 — Split Train into 10 Time-Ordered Folds by TransactionDT

# Sort by TransactionDT ascending
train_sorted = train_features.sort_values('TransactionDT').reset_index(drop=True)

n = len(train_sorted)
fold_size = n // 10

folds = {}
for i in range(10):
    start = i * fold_size
    end = (i + 1) * fold_size if i < 9 else n
    folds[i + 1] = train_sorted.iloc[start:end].index.tolist()

# Verify each fold
for i in range(1, 11):
    idx = folds[i]
    fold_df = train_sorted.loc[idx]
    print(
        f"T{i}: {len(idx)} transactions | "
        f"DT {fold_df['TransactionDT'].min()} ~ {fold_df['TransactionDT'].max()} | "
        f"fraud rate {fold_df['isFraud'].mean():.2%}"
    )

# Sanity check: no overlap between folds
all_indices = [idx for fold in folds.values() for idx in fold]
assert len(all_indices) == len(set(all_indices)), "Overlap detected between folds!"

# Sanity check: folds are time-ordered
for i in range(1, 10):
    max_dt_current = train_sorted.loc[folds[i], 'TransactionDT'].max()
    min_dt_next = train_sorted.loc[folds[i + 1], 'TransactionDT'].min()
    assert max_dt_current <= min_dt_next, f"Time ordering violated between T{i} and T{i+1}!"

print("Sanity checks passed: no overlap, folds are time-ordered.")




T1: 59054 transactions | DT 86400 ~ 1360999 | fraud rate 2.76%
T2: 59054 transactions | DT 1361005 ~ 2310138 | fraud rate 2.02%
T3: 59054 transactions | DT 2310165 ~ 3864159 | fraud rate 3.73%
T4: 59054 transactions | DT 3864166 ~ 5592303 | fraud rate 4.29%
T5: 59054 transactions | DT 5592304 ~ 7306520 | fraud rate 3.96%
T6: 59054 transactions | DT 7306535 ~ 8745772 | fraud rate 3.55%
T7: 59054 transactions | DT 8745798 ~ 10437996 | fraud rate 4.32%
T8: 59054 transactions | DT 10438003 ~ 12192842 | fraud rate 3.49%
T9: 59054 transactions | DT 12192900 ~ 13990904 | fraud rate 3.13%
T10: 59054 transactions | DT 13990941 ~ 15811131 | fraud rate 3.75%
Sanity checks passed: no overlap, folds are time-ordered.


In [4]:
# Block 4 — Build Edges: Connect Transactions Sharing Same Attribute Values

def build_edges(df, edge_cols, min_weight=2, max_group=100):
    edges = []

    for col in edge_cols:
        if col not in df.columns or df[col].isna().all():
            continue

        groups = df[col].dropna().groupby(df[col]).groups

        for value, idx_list in groups.items():
            idx_list = list(idx_list)
            if len(idx_list) < 2:
                continue
            if len(idx_list) > max_group:
                idx_list = idx_list[-max_group:]  # keep most recent

            for i in range(len(idx_list)):
                for j in range(i + 1, len(idx_list)):
                    edges.append((idx_list[i], idx_list[j]))

    if not edges:
        return pd.DataFrame(columns=['src', 'dst', 'weight'])

    edge_df = pd.DataFrame(edges, columns=['src', 'dst'])
    edge_df = (
        edge_df.groupby(['src', 'dst'])
        .size()
        .reset_index(name='weight')
    )
    edge_df = edge_df[edge_df['weight'] >= min_weight].reset_index(drop=True)

    print(f"Total edges: {len(edge_df):,} | "
          f"Avg weight: {edge_df['weight'].mean():.2f} | "
          f"Max weight: {edge_df['weight'].max()}")
    return edge_df


# Test on T1
test_edges = build_edges(train_sorted.loc[folds[1]], edge_cols)
print(f"T1 edge count: {len(test_edges):,}")
print(f"Memory estimate: {len(test_edges) * 16 / 1e9:.2f} GB")

Total edges: 483,654 | Avg weight: 2.32 | Max weight: 8
T1 edge count: 483,654
Memory estimate: 0.01 GB


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")

PyTorch: 2.4.1+cu124
CUDA: True
Device: NVIDIA L4


In [6]:
# ============================================================
# Block 5 — FraudGCN using PyTorch Geometric
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.utils import from_scipy_sparse_matrix
import scipy.sparse as sp
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class FraudGCN(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, emb_dim=32, dropout=0.3):
        super().__init__()
        self.dropout = dropout

        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, emb_dim)

        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.bn3 = nn.BatchNorm1d(emb_dim)

        self.classifier = nn.Linear(emb_dim, 1)

    def forward(self, x, edge_index, edge_weight=None):
        # layer 1
        h = self.conv1(x, edge_index, edge_weight)
        h = self.bn1(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        # layer 2
        h = self.conv2(h, edge_index, edge_weight)
        h = self.bn2(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        # layer 3 → embeddings
        emb = self.conv3(h, edge_index, edge_weight)
        emb = self.bn3(emb)
        emb = F.relu(emb)

        # fraud head
        out = self.classifier(emb)

        return emb, out


def build_pyg_graph(edge_df, n_nodes, txn_to_node):
    """
    Convert edge_df to PyG format edge_index and edge_weight
    """
    src = [txn_to_node[s] for s in edge_df['src'] if s in txn_to_node]
    dst = [txn_to_node[d] for d in edge_df['dst'] if d in txn_to_node]
    weights = edge_df['weight'].tolist()[:len(src)]

    # bidirectional
    rows = src + dst
    cols = dst + src
    vals = weights + weights

    # build scipy sparse for normalisation
    A = sp.coo_matrix(
        (vals, (rows, cols)),
        shape=(n_nodes, n_nodes),
        dtype=np.float32
    )

    # add self loops
    A = A + sp.eye(n_nodes, dtype=np.float32)

    # degree normalisation D^(-1/2) A D^(-1/2)
    rowsum    = np.array(A.sum(axis=1)).flatten()
    d_inv_sqrt = np.power(rowsum, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    D_inv_sqrt = sp.diags(d_inv_sqrt)
    A_norm     = D_inv_sqrt @ A @ D_inv_sqrt
    A_norm     = A_norm.tocoo()

    # convert to PyG format
    edge_index = torch.LongTensor(
        np.vstack([A_norm.row, A_norm.col])
    ).to(device)
    edge_weight = torch.FloatTensor(A_norm.data).to(device)

    print(f"  edge_index shape: {edge_index.shape}")
    print(f"  edge_weight shape: {edge_weight.shape}")

    return edge_index, edge_weight


# ── Verify ─────────────────────────────────────────────────
print("\n=== Block 5 — FraudGCN (PyG) ===")

dummy_input_dim = 10
dummy_n_nodes   = 100
dummy_n_edges   = 200

test_model = FraudGCN(
    input_dim  = dummy_input_dim,
    hidden_dim = 128,
    emb_dim    = 32
).to(device)

dummy_edge_index  = torch.randint(0, dummy_n_nodes, (2, dummy_n_edges)).to(device)
dummy_edge_weight = torch.ones(dummy_n_edges).to(device)
dummy_X           = torch.randn(dummy_n_nodes, dummy_input_dim).to(device)

emb, out = test_model(dummy_X, dummy_edge_index, dummy_edge_weight)

print(f"Input shape:     {dummy_X.shape}")
print(f"Embedding shape: {emb.shape}  ← should be ({dummy_n_nodes}, 32)")
print(f"Output shape:    {out.shape}  ← should be ({dummy_n_nodes}, 1)")
print(f"\n✅ FraudGCN (PyG) ready!")

Using device: cuda

=== Block 5 — FraudGCN (PyG) ===
Input shape:     torch.Size([100, 10])
Embedding shape: torch.Size([100, 32])  ← should be (100, 32)
Output shape:    torch.Size([100, 1])  ← should be (100, 1)

✅ FraudGCN (PyG) ready!


In [7]:
# ============================================================
# Block 6 — Single Round Training Function (PyG version)
# ============================================================

from sklearn.preprocessing import StandardScaler

def train_gcn_round(
    train_df, folds,
    graph_fold_ids, supervised_fold_ids, predict_fold_id,
    feature_cols, edge_cols,
    emb_dim=32, hidden_dim=128, epochs=300, lr=0.05,
    init_weights=None  # ← chain weights
):
    print(f"\n  Graph folds:      {graph_fold_ids}")
    print(f"  Supervised folds: {supervised_fold_ids}")
    print(f"  Predict fold:     {predict_fold_id}")

    # ── Step 1 — collect window transactions ──────────────
    all_idx     = sum([folds[fid] for fid in graph_fold_ids], [])
    window_df   = train_df.loc[all_idx].copy()
    n_nodes     = len(window_df)
    txn_to_node = {txn: i for i, txn in enumerate(window_df.index)}

    print(f"  Window size: {n_nodes:,} nodes")

    # ── Step 2 — node features ────────────────────────────
    scaler   = StandardScaler()
    X_np     = window_df[feature_cols].apply(
        pd.to_numeric, errors='coerce'
    ).fillna(0).values.astype(np.float32)
    X_scaled = scaler.fit_transform(X_np)
    X        = torch.FloatTensor(X_scaled).to(device)

    print(f"  Node features: {X.shape}")

    # ── Step 3 — build PyG graph ──────────────────────────
    print(f"  Building edges...")
    edge_df = build_edges(window_df, edge_cols, min_weight=2, max_group=100)

    if len(edge_df) == 0:
        print("  ⚠️ No edges — returning zero embeddings")
        predict_txns = folds[predict_fold_id]
        return pd.DataFrame(
            np.zeros((len(predict_txns), emb_dim)),
            index   = predict_txns,
            columns = [f'emb_{i}' for i in range(emb_dim)]
        ), None

    edge_index, edge_weight = build_pyg_graph(edge_df, n_nodes, txn_to_node)

    # ── Step 4 — supervised nodes and labels ──────────────
    sup_node_idx = []
    sup_labels   = []

    for fid in supervised_fold_ids:
        for txn in folds[fid]:
            if txn in txn_to_node:
                sup_node_idx.append(txn_to_node[txn])
                sup_labels.append(window_df.loc[txn, 'isFraud'])

    sup_node_idx        = torch.LongTensor(sup_node_idx).to(device)
    sup_labels          = torch.FloatTensor(sup_labels).to(device)

    pred_node_idx = [
        txn_to_node[txn]
        for txn in folds[predict_fold_id]
        if txn in txn_to_node
    ]

    print(f"  Supervised nodes: {len(sup_node_idx):,}")
    print(f"  Predict nodes:    {len(pred_node_idx):,}")
    print(f"  Fraud rate:       {sup_labels.mean():.2%}")

    # ── Step 5 — train GCN ────────────────────────────────
    model = FraudGCN(
        input_dim  = X.shape[1],
        hidden_dim = hidden_dim,
        emb_dim    = emb_dim
    ).to(device)

    # load previous round weights if available
    if init_weights is not None:
        model.load_state_dict(init_weights)
        print("  Initialised from previous round weights ✅")
    else:
        print("  Random initialisation (Round 1)")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=60, gamma=0.5
    )

    n_pos      = sup_labels.sum()
    n_neg      = len(sup_labels) - n_pos
    pos_weight = torch.FloatTensor(
        [n_neg / (n_pos + 1e-8)]
    ).to(device)

    best_loss = float('inf')
    best_emb  = None

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        emb, out = model(X, edge_index, edge_weight)

        loss = F.binary_cross_entropy_with_logits(
            out[sup_node_idx].squeeze(),
            sup_labels,
            pos_weight=pos_weight
        )

        loss.backward()
        optimizer.step()
        scheduler.step()

        if loss.item() < best_loss:
            best_loss = loss.item()
            model.eval()
            with torch.no_grad():
                best_emb, _ = model(X, edge_index, edge_weight)
            best_emb = best_emb.cpu().numpy()

        if epoch % 20 == 0:
            print(f"  Epoch {epoch:3d} | loss: {loss.item():.4f}")

    print(f"  ✅ Best loss: {best_loss:.4f}")

    # ── Step 6 — extract predict fold embeddings ──────────
    predict_txns     = [
        txn for txn in folds[predict_fold_id]
        if txn in txn_to_node
    ]
    predict_node_idx = [txn_to_node[t] for t in predict_txns]
    predict_emb      = best_emb[predict_node_idx]

    emb_df = pd.DataFrame(
        predict_emb,
        index   = predict_txns,
        columns = [f'emb_{i}' for i in range(emb_dim)]
    )

    print(f"  Embeddings shape: {emb_df.shape}")

    return emb_df, model  # ← return model for chain weights


print("✅ Block 6 — train_gcn_round() (PyG) ready!")

✅ Block 6 — train_gcn_round() (PyG) ready!


In [8]:
# ── 从 feature_cols 里去掉 edge_only_cols ──────────────────
# edge_only_cols = ['P_emaildomain', 'R_emaildomain']

# feature_cols = [c for c in feature_cols if c not in edge_only_cols]

# print(f"Removed: {edge_only_cols}")
# print(f"feature_cols now: {len(feature_cols)}")

# ── encode 剩余 object 列 ───────────────────────────────────
from sklearn.preprocessing import LabelEncoder

object_cols_remaining = [
    c for c in feature_cols 
    if train_features[c].dtype == 'object'
]
print(f"\nObject cols to encode: {len(object_cols_remaining)}")
print(object_cols_remaining)

le = LabelEncoder()

for col in object_cols_remaining:
    train_features[col] = train_features[col].astype(str).replace('nan', 'unknown')
    test_features[col]  = test_features[col].astype(str).replace('nan', 'unknown')
    
    known = list(train_features[col].unique()) + ['unknown']
    le.fit(known)
    
    train_features[col] = le.transform(train_features[col])
    test_features[col]  = le.transform(test_features[col])

# ── 验证 ────────────────────────────────────────────────────
object_remaining = [c for c in feature_cols if train_features[c].dtype == 'object']
print(f"\nRemaining object cols: {len(object_remaining)}")
print("✅ All numeric!" if len(object_remaining) == 0 else f"❌ Still object: {object_remaining}")

# ── 更新 train_sorted ────────────────────────────────────────
train_sorted = train_features.sort_values('TransactionDT').reset_index(drop=True)
print(f"✅ train_sorted updated: {train_sorted.shape}")


Object cols to encode: 0
[]

Remaining object cols: 0
✅ All numeric!
✅ train_sorted updated: (590540, 638)


In [9]:
# ============================================================
# Block 7 — Run All 5 Rounds (With Chain Logic)
# ============================================================

import time

emb_cols    = [f'emb_{i}' for i in range(32)]
rounds = [
    {'graph': [1,2,3,4,5,6],  'supervised': [1,2,3,4,5],  'predict': 6},
    {'graph': [2,3,4,5,6,7],  'supervised': [2,3,4,5,6],  'predict': 7},
    {'graph': [3,4,5,6,7,8],  'supervised': [3,4,5,6,7],  'predict': 8},
    {'graph': [4,5,6,7,8,9],  'supervised': [4,5,6,7,8],  'predict': 9},
    {'graph': [5,6,7,8,9,10], 'supervised': [5,6,7,8,9],  'predict': 10},
]

oof_embeddings = []
total_start    = time.time()
prev_weights   = None  # ← 追踪上一轮权重

for i, r in enumerate(rounds):
    print(f"\n{'='*55}")
    print(f"  ROUND {i+1}/5  →  predicting fold T{r['predict']}")
    print(f"{'='*55}")

    round_start = time.time()

    emb_df, model = train_gcn_round(  # ← 接收 model
        train_df            = train_sorted,
        folds               = folds,
        graph_fold_ids      = r['graph'],
        supervised_fold_ids = r['supervised'],
        predict_fold_id     = r['predict'],
        feature_cols        = feature_cols,
        edge_cols           = edge_cols,
        emb_dim             = 32,
        hidden_dim          = 128,
        epochs              = 500,
        lr                  = 0.05,
        init_weights        = prev_weights  # ← 传入上一轮权重
    )

    # ← 保存当前轮权重给下一轮
    prev_weights = model.state_dict()

    oof_embeddings.append(emb_df)

    save_path = f'/workspace/Embedding_res_chain/oof_emb_round{i+1}.csv'
    emb_df.to_csv(save_path)

    # ← Round 5 结束后保存权重供 Block 9 使用
    if i == 4:
        torch.save(
            prev_weights,
            '/workspace/Embedding_res_chain/gcn_round5_weights.pt'
        )
        print("✅ Round 5 weights saved!")

    round_time = time.time() - round_start
    print(f"\n  Round {i+1} done ✅")
    print(f"  Embeddings: {emb_df.shape}")
    print(f"  Saved to:   {save_path}")
    print(f"  Time taken: {round_time/60:.1f} mins")

# ── Combine ───────────────────────────────────────────────
oof_emb_df = pd.concat(oof_embeddings)
assert oof_emb_df.index.duplicated().sum() == 0

T6_T10_idx = sum([folds[i] for i in range(6, 11)], [])
coverage   = len(set(oof_emb_df.index) & set(T6_T10_idx)) / len(T6_T10_idx) * 100

oof_emb_df.to_csv('/workspace/Embedding_res_chain/oof_embeddings.csv')

total_time = time.time() - total_start
print(f"\n=== Block 7 Complete (Chain Logic) ===")
print(f"OOF embeddings shape:  {oof_emb_df.shape}")
print(f"Covers T6~T10:         {coverage:.1f}%")
print(f"Total time:            {total_time/60:.1f} mins")
print(f"Saved: /workspace/Embedding_res_chain/oof_embeddings.csv")


  ROUND 1/5  →  predicting fold T6

  Graph folds:      [1, 2, 3, 4, 5, 6]
  Supervised folds: [1, 2, 3, 4, 5]
  Predict fold:     6
  Window size: 354,324 nodes
  Node features: torch.Size([354324, 631])
  Building edges...
Total edges: 2,662,906 | Avg weight: 2.35 | Max weight: 8
  edge_index shape: torch.Size([2, 5680136])
  edge_weight shape: torch.Size([5680136])
  Supervised nodes: 295,270
  Predict nodes:    59,054
  Fraud rate:       3.35%
  Random initialisation (Round 1)
  Epoch   0 | loss: 1.3639
  Epoch  20 | loss: 0.9053
  Epoch  40 | loss: 0.8009
  Epoch  60 | loss: 0.6998
  Epoch  80 | loss: 0.6371
  Epoch 100 | loss: 0.5767
  Epoch 120 | loss: 0.5239
  Epoch 140 | loss: 0.4937
  Epoch 160 | loss: 0.4734
  Epoch 180 | loss: 0.4490
  Epoch 200 | loss: 0.4354
  Epoch 220 | loss: 0.4218
  Epoch 240 | loss: 0.4129
  Epoch 260 | loss: 0.4043
  Epoch 280 | loss: 0.3951
  Epoch 300 | loss: 0.3909
  Epoch 320 | loss: 0.3943
  Epoch 340 | loss: 0.3844
  Epoch 360 | loss: 0.3784


In [10]:
# ============================================================
# Block 8 — Sanity Check
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

emb_cols = [f'emb_{i}' for i in range(32)]

print("=" * 55)
print("  Block 8 — Sanity Check")
print("=" * 55)

# ── Check 1 — Basic Statistics ────────────────────────────
print("\n[1] Embedding Statistics")
print(oof_emb_df[emb_cols].describe().round(4))

# flag if all values identical → GCN not learning
std_per_dim = oof_emb_df[emb_cols].std()
dead_dims   = (std_per_dim < 0.001).sum()
print(f"\nDead dimensions (std < 0.001): {dead_dims}/32")
if dead_dims > 16:
    print("⚠️  More than half dims are dead — GCN may not be learning")
else:
    print("✅ Embedding dimensions are alive")

# ── Check 2 — Fraud vs Non-Fraud Separation ───────────────
print("\n[2] Fraud vs Non-Fraud Embedding Separation")

# join with labels
oof_with_labels = oof_emb_df.copy()
oof_with_labels['isFraud'] = train_sorted.loc[
    oof_emb_df.index, 'isFraud'
].values

fraud_mean    = oof_with_labels[
    oof_with_labels['isFraud'] == 1
][emb_cols].mean()

nonfraud_mean = oof_with_labels[
    oof_with_labels['isFraud'] == 0
][emb_cols].mean()

diff = abs(fraud_mean - nonfraud_mean)
diff_sorted = diff.sort_values(ascending=False)

print(f"\nTop 10 most discriminative embedding dimensions:")
print(diff_sorted.head(10).round(4))
print(f"\nMean separation: {diff.mean():.4f}")
print(f"Max separation:  {diff.max():.4f}")

if diff.mean() < 0.01:
    print("⚠️  Very low separation — GCN not learning fraud patterns")
elif diff.mean() < 0.05:
    print("⚠️  Low separation — GCN learning weakly")
else:
    print("✅ Good separation — GCN learned fraud patterns")

# ── Check 3 — Quick AUC from Embeddings Only ──────────────
print("\n[3] Quick AUC — Embeddings Only (Logistic Regression)")

# T6~T9 train, T10 validate
T6_T9_idx = []
for fid in range(6, 10):
    T6_T9_idx.extend(folds[fid])

T10_idx = folds[10]

# filter to only transactions in oof_emb_df
T6_T9_in_oof = [i for i in T6_T9_idx if i in oof_emb_df.index]
T10_in_oof   = [i for i in T10_idx   if i in oof_emb_df.index]

X_tr  = oof_emb_df.loc[T6_T9_in_oof, emb_cols]
y_tr  = train_sorted.loc[T6_T9_in_oof, 'isFraud']
X_val = oof_emb_df.loc[T10_in_oof,   emb_cols]
y_val = train_sorted.loc[T10_in_oof,  'isFraud']

print(f"Train size: {len(X_tr):,} | Val size: {len(X_val):,}")
print(f"Fraud rate train: {y_tr.mean():.2%} | val: {y_val.mean():.2%}")

# scale embeddings
scaler   = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_tr)
X_val_sc = scaler.transform(X_val)

# logistic regression
lr_model = LogisticRegression(
    max_iter    = 1000,
    class_weight = 'balanced',
    random_state = 42
)
lr_model.fit(X_tr_sc, y_tr)
pred = lr_model.predict_proba(X_val_sc)[:, 1]
auc  = roc_auc_score(y_val, pred)

print(f"\nEmbedding-only AUC: {auc:.4f}")

if auc < 0.55:
    print("❌ AUC near random — GCN not learning, need to investigate")
elif auc < 0.65:
    print("⚠️  Weak signal — GCN learning something but limited")
elif auc < 0.75:
    print("✅ Moderate signal — GCN useful as supplementary feature")
else:
    print("✅✅ Strong signal — GCN embeddings are highly informative!")

# ── Check 4 — Distribution Per Round ─────────────────────
print("\n[4] Embedding Distribution Per Round (T6~T10)")

for fid in range(6, 11):
    fold_idx = [i for i in folds[fid] if i in oof_emb_df.index]
    fold_emb = oof_emb_df.loc[fold_idx, emb_cols]
    print(f"  T{fid}: mean={fold_emb.mean().mean():.4f} | "
          f"std={fold_emb.std().mean():.4f} | "
          f"n={len(fold_emb):,}")

print("\n  → Similar mean/std across folds = consistent distribution ✅")
print("  → Very different values = distribution shift ⚠️")

# ── Summary ───────────────────────────────────────────────
print(f"\n{'='*55}")
print("  Sanity Check Summary")
print(f"{'='*55}")
print(f"  Dead dimensions:    {dead_dims}/32")
print(f"  Mean separation:    {diff.mean():.4f}")
print(f"  Embedding AUC:      {auc:.4f}")
print(f"  OOF shape:          {oof_emb_df.shape}")

if auc > 0.65 and dead_dims < 16 and diff.mean() > 0.01:
    print("\n✅ GCN embeddings look useful — proceed to Block 9")
else:
    print("\n⚠️  Consider tuning: more epochs, higher lr, or check graph structure")

  Block 8 — Sanity Check

[1] Embedding Statistics
             emb_0        emb_1        emb_2        emb_3        emb_4  \
count  295270.0000  295270.0000  295270.0000  295270.0000  295270.0000   
mean        0.1814       0.7028       0.7548       0.1715       0.0113   
std         0.4290       1.3334       1.5166       0.9357       0.1672   
min         0.0000       0.0000       0.0000       0.0000       0.0000   
25%         0.0000       0.1209       0.2178       0.0000       0.0000   
50%         0.0000       0.4739       0.5386       0.0000       0.0000   
75%         0.2180       0.9208       0.9357       0.0947       0.0000   
max        16.8901      50.2539      40.3333      26.2999      25.9135   

             emb_5        emb_6        emb_7        emb_8        emb_9  ...  \
count  295270.0000  295270.0000  295270.0000  295270.0000  295270.0000  ...   
mean        0.3308       0.0000       0.0849       0.0858       0.0150  ...   
std         0.4596       0.0065       0.2402 

In [13]:
import time
import gc

print("=" * 55)
print("  Block 9 — Test Embeddings (Fixed T6~T10 Window)")
print("=" * 55)

gc.collect()
torch.cuda.empty_cache()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# ── Step 1 — Get model input_dim ──────────────────────────
checkpoint      = torch.load('/workspace/Embedding_res/gcn_round5_weights.pt')
model_input_dim = checkpoint['conv1.lin.weight'].shape[1]
feature_cols_b9 = feature_cols[:model_input_dim]


  Block 9 — Test Embeddings (Fixed T6~T10 Window)
GPU free: 20.5 GB


/tmp/ipykernel_335/751710802.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint      = torch.load('/workspace/Embedding_res/gcn_round5_weights.pt')


In [18]:
model_input_dim

631

In [9]:
# ============================================================
# Block 9 — Test Embeddings (Fixed Window T6~T10)
# ============================================================

import time
import gc

print("=" * 55)
print("  Block 9 — Test Embeddings (Fixed T6~T10 Window)")
print("=" * 55)

gc.collect()
torch.cuda.empty_cache()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# ── Step 1 — Get model input_dim ──────────────────────────
checkpoint      = torch.load('/workspace/Embedding_res_chain/gcn_round5_weights.pt')
model_input_dim = checkpoint['conv1.lin.weight'].shape[1]
feature_cols_b9 = feature_cols
emb_cols = [f'emb_{i}' for i in range(32)]
print(f"Model input_dim: {model_input_dim}")

# ── Step 2 — Prepare T6~T10 (fixed) ───────────────────────
print("\n[1] Preparing fixed train context T6~T10...")

T6_T10_idx   = sum([folds[i] for i in range(6, 11)], [])
train_context = train_sorted.loc[T6_T10_idx].copy()
print(f"Train context (T6~T10): {len(train_context):,}")

# ── Step 3 — Sort and split test ──────────────────────────
print("\n[2] Splitting test into 10 chunks...")

test_features_sorted = test_features.sort_values(
    'TransactionDT'
).reset_index(drop=True)

n_test     = len(test_features_sorted)
chunk_size = n_test // 10

test_chunks = {}
for i in range(10):
    start = i * chunk_size
    end   = (i+1) * chunk_size if i < 9 else n_test
    test_chunks[i+1] = test_features_sorted.iloc[start:end].copy()
    test_chunks[i+1]['isFraud'] = -1
    print(f"  test{i+1}: {len(test_chunks[i+1]):,}")

# ── Step 4 — Load model ───────────────────────────────────
print("\n[3] Loading Round 5 weights...")

model = FraudGCN(
    input_dim  = model_input_dim,
    hidden_dim = 128,
    emb_dim    = 32
).to(device)

model.load_state_dict(checkpoint)
model.eval()
print("✅ Model loaded")

# ── Step 5 — Process each test chunk ──────────────────────
all_test_emb = []
total_start  = time.time()

for chunk_id in range(1, 11):
    print(f"\n{'='*50}")
    print(f"  Test{chunk_id}/10 with T6~T10 context")
    print(f"{'='*50}")

    gc.collect()
    torch.cuda.empty_cache()

    # combine fixed train context + this test chunk
    test_chunk = test_chunks[chunk_id]
    
    extra_cols  = ['isFraud', 'TransactionDT']
    common_cols = list(dict.fromkeys(feature_cols_b9 + extra_cols))
    common_cols = [c for c in common_cols 
                   if c in train_context.columns and c in test_chunk.columns]

    graph_df = pd.concat(
        [train_context[common_cols], test_chunk[common_cols]],
        ignore_index=True
    )
    
    # mark test nodes
    graph_df['is_test'] = False
    graph_df.iloc[len(train_context):, graph_df.columns.get_loc('is_test')] = True

    n_nodes     = len(graph_df)
    txn_to_node = {idx: i for i, idx in enumerate(graph_df.index)}

    print(f"  Graph nodes: {n_nodes:,}")
    print(f"  Test nodes:  {len(test_chunk):,}")

    # node features
    scaler   = StandardScaler()
    X_np     = graph_df[feature_cols_b9].apply(
        pd.to_numeric, errors='coerce'
    ).fillna(0).values.astype(np.float32)
    X_scaled = scaler.fit_transform(X_np)
    X        = torch.FloatTensor(X_scaled).to(device)

    # build edges
    edge_df = build_edges(
        graph_df, edge_cols,
        min_weight = 2,
        max_group  = 100
    )

    print(f"  Edges: {len(edge_df):,}")
    print(f"  GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

    if len(edge_df) == 0:
        print("  ⚠️ No edges — using zero embeddings")
        test_idx = graph_df[graph_df['is_test']].index.tolist()
        zero_emb = pd.DataFrame(
            np.zeros((len(test_idx), 32)),
            index   = test_idx,
            columns = emb_cols
        )
        all_test_emb.append(zero_emb)
        continue

    edge_index, edge_weight = build_pyg_graph(
        edge_df, n_nodes, txn_to_node
    )

    # forward pass only
    with torch.no_grad():
        emb, _ = model(X, edge_index, edge_weight)

    emb_np = emb.cpu().numpy()

    # extract test embeddings
    test_idx_in_graph = graph_df[graph_df['is_test']].index.tolist()
    test_node_idx     = [txn_to_node[i] for i in test_idx_in_graph 
                         if i in txn_to_node]

    chunk_emb_df = pd.DataFrame(
        emb_np[test_node_idx],
        index   = test_idx_in_graph,
        columns = emb_cols
    )

    all_test_emb.append(chunk_emb_df)
    print(f"  ✅ test{chunk_id} done: {chunk_emb_df.shape}")

    # cleanup
    del X, edge_index, edge_weight, emb, emb_np
    gc.collect()
    torch.cuda.empty_cache()

# ── Step 6 — Combine and save ─────────────────────────────
print(f"\n{'='*50}")
print("  Combining all chunks...")

test_emb_df = pd.concat(all_test_emb)
print(f"Test embeddings shape: {test_emb_df.shape}")
print(f"Total time: {(time.time()-total_start)/60:.1f} mins")

test_emb_df.to_csv('/workspace/Embedding_res_chain/test_embeddings_fixed.csv')

print(f"\n{'='*55}")
print("  Block 9 Complete (Fixed Window)")
print(f"{'='*55}")
print(f"  Test embeddings: {test_emb_df.shape}")
print(f"  Saved: /workspace/Embedding_res_chain/test_embeddings_fixed.csv ✅")

  Block 9 — Test Embeddings (Fixed T6~T10 Window)
GPU free: 23.4 GB
Model input_dim: 631

[1] Preparing fixed train context T6~T10...


/tmp/ipykernel_979/424503175.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint      = torch.load('/workspace/Embedding_res_chain/gcn_round5_weights.pt')


Train context (T6~T10): 295,270

[2] Splitting test into 10 chunks...
  test1: 50,669
  test2: 50,669
  test3: 50,669
  test4: 50,669
  test5: 50,669
  test6: 50,669
  test7: 50,669
  test8: 50,669
  test9: 50,669
  test10: 50,670

[3] Loading Round 5 weights...
✅ Model loaded

  Test1/10 with T6~T10 context
  Graph nodes: 345,939
  Test nodes:  50,669
Total edges: 7,410 | Avg weight: 2.21 | Max weight: 3
  Edges: 7,410
  GPU free: 22.5 GB
  edge_index shape: torch.Size([2, 360759])
  edge_weight shape: torch.Size([360759])
  ✅ test1 done: (50669, 32)

  Test2/10 with T6~T10 context
  Graph nodes: 345,939
  Test nodes:  50,669
Total edges: 7,052 | Avg weight: 2.28 | Max weight: 3
  Edges: 7,052
  GPU free: 22.5 GB
  edge_index shape: torch.Size([2, 360043])
  edge_weight shape: torch.Size([360043])
  ✅ test2 done: (50669, 32)

  Test3/10 with T6~T10 context
  Graph nodes: 345,939
  Test nodes:  50,669
Total edges: 7,445 | Avg weight: 2.22 | Max weight: 3
  Edges: 7,445
  GPU free: 22.5

In [11]:
################################

import gc
import torch

# 删除所有大型 GPU 对象
for obj_name in ['model', 'X', 'edge_index', 'edge_weight', 'A_norm']:
    if obj_name in dir():
        del globals()[obj_name]

gc.collect()
torch.cuda.empty_cache()

print(f"GPU memory free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

GPU memory free: 23.4 GB


In [ ]:
# ============================================================
# Block 9 — Use train_gcn_round for test (extend chain)
# ============================================================

import time
import gc

print("=" * 55)
print("  Block 9 — Test as Round 6 (Chain Extension)")
print("=" * 55)

gc.collect()
torch.cuda.empty_cache()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# ── Step 1 — Combine train + test into one DataFrame ──────
print("\n[1] Combining train + test...")

test_features['isFraud'] = -1  # placeholder (not used)

# need same columns
extra_cols  = ['isFraud', 'TransactionDT']
common_cols = list(dict.fromkeys(feature_cols + extra_cols))
common_cols = [c for c in common_cols 
               if c in train_sorted.columns and c in test_features.columns]

combined_df = pd.concat([
    train_sorted[common_cols],
    test_features[common_cols]
], ignore_index=True)

print(f"Combined shape: {combined_df.shape}")

# ── Step 2 — Create extended folds ────────────────────────
# train folds use original indices (0 to len(train_sorted)-1)
# test fold uses indices len(train_sorted) onwards

extended_folds = {}
for fid, idx_list in folds.items():
    extended_folds[fid] = idx_list  # train folds unchanged

# add test as a new fold
test_start = len(train_sorted)
test_end   = len(combined_df)
extended_folds['test'] = list(range(test_start, test_end))

print(f"Train folds 1~10: each ~{len(folds[1]):,}")
print(f"Test fold:        {len(extended_folds['test']):,}")

# ── Step 3 — Load Round 5 weights ─────────────────────────
print("\n[2] Loading Round 5 weights...")

checkpoint   = torch.load('/workspace/Embedding_res_chain/gcn_round5_weights.pt')
prev_weights = checkpoint
print("✅ Round 5 weights loaded")

# ── Step 4 — Run as "Round 6" ─────────────────────────────
print("\n[3] Running test as Round 6 (chain extension)...")

start = time.time()

# important: lower epochs and lr for fine-tuning
emb_df, model = train_gcn_round(
    train_df            = combined_df,
    folds               = extended_folds,
    graph_fold_ids      = [6, 7, 8, 9, 10, 'test'],  # include test in graph
    supervised_fold_ids = [6, 7, 8, 9, 10],          # supervise on train only
    predict_fold_id     = 'test',                     # predict test
    feature_cols        = feature_cols,
    edge_cols           = edge_cols,
    emb_dim             = 32,
    hidden_dim          = 128,
    epochs              = 100,                        # ← fewer epochs
    lr                  = 0.005,                      # ← lower lr
    init_weights        = prev_weights                # ← chain from Round 5
)

print(f"\n✅ Done | Time: {(time.time()-start)/60:.1f} mins")
print(f"Test embeddings: {emb_df.shape}")

# ── Step 5 — Save ─────────────────────────────────────────
emb_df.to_csv('/workspace/Embedding_res_chain/test_embeddings_chain_all_train.csv')

print(f"\n{'='*55}")
print("  Block 9 Complete (Chain Extension)")
print(f"{'='*55}")
print(f"  Test embeddings: {emb_df.shape}")
print(f"  Saved: test_embeddings_chain_all_train.csv ✅")

  Block 9 — Test as Round 6 (Chain Extension)
GPU free: 23.4 GB

[1] Combining train + test...


/tmp/ipykernel_361/2548686310.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_features['isFraud'] = -1  # placeholder (not used)


Combined shape: (1097231, 632)
Train folds 1~10: each ~59,054
Test fold:        506,691

[2] Loading Round 5 weights...
✅ Round 5 weights loaded

[3] Running test as Round 6 (chain extension)...

  Graph folds:      [6, 7, 8, 9, 10, 'test']
  Supervised folds: [6, 7, 8, 9, 10]
  Predict fold:     test


/tmp/ipykernel_361/2548686310.py:53: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint   = torch.load('/workspace/Embedding_res_chain/gcn_round5_weights.pt')


  Window size: 801,961 nodes
